#### Importing Required Libararies:

In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay
)

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_validate
import time
from sklearn.model_selection import GridSearchCV

#### Loading data and preprocessor:

In [ ]:
X_train = joblib.load("../data/X_train.pkl")
X_test = joblib.load("../data/X_test.pkl")
y_train = joblib.load("../data/y_train.pkl")
y_test = joblib.load("../data/y_test.pkl")

preprocessor = joblib.load("../outputs/models/preprocessor.pkl")

# Application and Evaluation of different ML models:

I did that because it will prevent the repition of code.

#### Perform Cross Validation:

This function perform cross-validation on the training data.

In [ ]:
def perform_cross_validation(pipeline, X_train, y_train):

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    cv_results = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=[
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc"
        ],
        n_jobs=-1,
        return_train_score=False
    )

    cv_results_df = pd.DataFrame(cv_results)

    cv_results_df = cv_results_df.drop(
        columns=["fit_time", "score_time"]
    )

    return cv_results_df

#### Print Fold-wise Results:

This fuction print fold-wise cross-validation results.

In [ ]:
def print_cv_fold_results(cv_results):
    df = pd.DataFrame({
        "Fold": range(1, len(cv_results["test_accuracy"]) + 1),
        "Accuracy": cv_results["test_accuracy"],
        "Precision": cv_results["test_precision"],
        "Recall": cv_results["test_recall"],
        "F1 Score": cv_results["test_f1"],
        "ROC AUC": cv_results["test_roc_auc"]
    })

    return df

#### Print Average Cross Validation Results:

Print average cross-validation metrics.

In [ ]:
def print_cv_average_results(cv_results):
    cv_df = pd.DataFrame(cv_results)

    print(f"Accuracy  : {cv_df['test_accuracy'].mean():.4f}")
    print(f"Precision : {cv_df['test_precision'].mean():.4f}")
    print(f"Recall    : {cv_df['test_recall'].mean():.4f}")
    print(f"F1 Score  : {cv_df['test_f1'].mean():.4f}")
    print(f"ROC AUC   : {cv_df['test_roc_auc'].mean():.4f}")

#### Print Standard Deviation:

Print the standard deviation of cross-validation metrics.

In [ ]:
def print_cv_standard_deviation(cv_results):

    cv_df = pd.DataFrame(cv_results)


    print(f"Accuracy  : {cv_df['test_accuracy'].std():.4f}")
    print(f"Precision : {cv_df['test_precision'].std():.4f}")
    print(f"Recall    : {cv_df['test_recall'].std():.4f}")
    print(f"F1 Score  : {cv_df['test_f1'].std():.4f}")
    print(f"ROC AUC   : {cv_df['test_roc_auc'].std():.4f}")

#### Fit Pipeline:

This function train the pipeline and record training time.

In [ ]:
def fit_pipeline(pipeline, X_train, y_train):
    start_time = time.perf_counter()
    pipeline.fit(X_train, y_train)
    training_time = time.perf_counter() - start_time
    
    return pipeline, training_time

#### Predict:

This function generate predictions on both training and test sets.

In [ ]:
def predict_pipeline(pipeline, X_train, X_test):

    train_predictions = pipeline.predict(X_train)
    test_predictions = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):

        train_probabilities = pipeline.predict_proba(X_train)[:, 1]
        test_probabilities = pipeline.predict_proba(X_test)[:, 1]

    else:

        train_probabilities = None
        test_probabilities = None

    return (
        train_predictions,
        test_predictions,
        train_probabilities,
        test_probabilities
    )

#### Print Test Metrics:

In [ ]:
def print_test_metrics(y_test, test_predictions, test_probabilities):

    accuracy = accuracy_score(y_test, test_predictions)
    precision = precision_score(y_test, test_predictions)
    recall = recall_score(y_test, test_predictions)
    f1 = f1_score(y_test, test_predictions)

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }

    if test_probabilities is not None:
        roc_auc = roc_auc_score(y_test, test_probabilities)

        print(f"ROC AUC   : {roc_auc:.4f}")

        metrics["ROC AUC"] = roc_auc

    return metrics

#### Print Confusion Matrix:

This fuction plots confusion matrix.

In [ ]:
def print_confusion_matrix(y_test, test_predictions, model_name):

    fig, ax = plt.subplots(figsize=(2, 2))

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        test_predictions,
        ax=ax,
        colorbar=False
    )

    ax.set_title(f"{model_name} - Confusion Matrix")

    plt.tight_layout()
    plt.show()

#### Plot Feature importance:

This function only works for tree based models.

In [ ]:
def plot_feature_importance(pipeline, feature_names, model_name, top_n=None):
    
    # Extract the trained classifier
    model = pipeline.named_steps["classifier"]

    # Ensure the model supports feature importance
    if not hasattr(model, "feature_importances_"):
        raise ValueError(f"{model_name} does not support feature_importances_.")

    # Create dataframe
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": model.feature_importances_
    })

    # Sort by importance
    importance_df = importance_df.sort_values(
        by="Importance",
        ascending=False
    )

    # Keep only top N features if requested
    if top_n is not None:
        importance_df = importance_df.head(top_n)

    # Plot
    plt.figure(figsize=(10, 6))

    plt.barh(
        importance_df["Feature"],
        importance_df["Importance"]
    )

    plt.gca().invert_yaxis()

    plt.xlabel("Feature Importance")
    plt.ylabel("Feature")
    plt.title(f"{model_name} Feature Importance")

    plt.tight_layout()
    plt.show()

    return importance_df

#### Plot ROC Curve:

This function plot ROC curve if probabilities are available.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

def plot_roc_curve(y_test, test_probabilities, model_name):

    fig, ax = plt.subplots(figsize=(3, 3))

    RocCurveDisplay.from_predictions(
        y_test,
        test_probabilities,
        ax=ax
    )

    ax.set_title(f"{model_name} - ROC Curve")

    plt.tight_layout()
    plt.show()

#### Compare Train vs Test Performance:

Compare training and test performance to check for overfitting.

In [ ]:
def compare_train_vs_test(y_train, train_predictions, y_test, test_predictions):

    # Training metrics
    train_accuracy = accuracy_score(y_train, train_predictions)
    train_precision = precision_score(y_train, train_predictions)
    train_recall = recall_score(y_train, train_predictions)
    train_f1 = f1_score(y_train, train_predictions)

    # Test metrics
    test_accuracy = accuracy_score(y_test, test_predictions)
    test_precision = precision_score(y_test, test_predictions)
    test_recall = recall_score(y_test, test_predictions)
    test_f1 = f1_score(y_test, test_predictions)

    comparison_df = pd.DataFrame({
        "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
        "Training": [
            train_accuracy,
            train_precision,
            train_recall,
            train_f1
        ],
        "Test": [
            test_accuracy,
            test_precision,
            test_recall,
            test_f1
        ]
    })

    comparison_df["Difference"] = abs(
        comparison_df["Training"] - comparison_df["Test"]
    )

    comparison_df[["Training", "Test", "Difference"]] = comparison_df[
        ["Training", "Test", "Difference"]
    ].round(4)

    return comparison_df

### 1. Random Forest:

In [ ]:
random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", RandomForestClassifier(
        random_state=42
    ))
])

##### Perform Cross Validation:

In [ ]:
cv_results = perform_cross_validation(
    pipeline=random_forest_pipeline,
    X_train=X_train,
    y_train=y_train
)

##### Print Fold-wise CV Results:

In [ ]:
print_cv_fold_results(cv_results)

#####  Print Average CV Results:

In [ ]:
print_cv_average_results(cv_results)

##### Print Standard Deviation:

In [ ]:
print_cv_standard_deviation(cv_results)

##### Train the Pipeline:

In [ ]:
random_forest_trained_pipeline, training_time = fit_pipeline(
    pipeline=random_forest_pipeline,
    X_train=X_train,
    y_train=y_train
)

print(f"\nTraining Time: {training_time:.4f} seconds")

In [ ]:
joblib.dump(random_forest_trained_pipeline, "../outputs/models/random_forest.pkl")

##### Make Predictions:

In [ ]:
(
    train_predictions,
    test_predictions,
    train_probabilities,
    test_probabilities
) = predict_pipeline(
    pipeline=random_forest_trained_pipeline,
    X_train=X_train,
    X_test=X_test
)

##### Feature importance:

In [ ]:
plot_feature_importance(
    pipeline=random_forest_trained_pipeline,
    feature_names=X_train.columns,
    model_name="Random Forest"
)

##### Print Test Metrics:

In [ ]:
metrics = print_test_metrics(
    y_test=y_test,
    test_predictions=test_predictions,
    test_probabilities=test_probabilities
)

##### Print Confusion Matrix:

In [ ]:
print_confusion_matrix(
    y_test=y_test,
    test_predictions=test_predictions,
    model_name="Random Forest"
)

##### Plot ROC Curve:

In [ ]:
plot_roc_curve(
    y_test=y_test,
    test_probabilities=test_probabilities,
    model_name="Random Forest"
)

##### Compare Training and Test Performance:

In [ ]:
compare_train_vs_test(
    y_train=y_train,
    train_predictions=train_predictions,
    y_test=y_test,
    test_predictions=test_predictions
)

### 2. Logistic Regression:

In [ ]:
logistic_regression_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [ ]:
cv_results = perform_cross_validation(
    pipeline=logistic_regression_pipeline ,
    X_train=X_train,
    y_train=y_train
)

In [ ]:
print_cv_fold_results(cv_results)

In [ ]:
print_cv_average_results(cv_results)

In [ ]:
print_cv_standard_deviation(cv_results)

In [ ]:
logistic_regression_trained_pipeline, training_time = fit_pipeline(
    pipeline=logistic_regression_pipeline,
    X_train=X_train,
    y_train=y_train
)

print(f"\nTraining Time: {training_time:.4f} seconds")


In [ ]:
joblib.dump(logistic_regression_trained_pipeline, "../outputs/models/logistic_regression.pkl")

In [ ]:
(
    train_predictions,
    test_predictions,
    train_probabilities,
    test_probabilities
) = predict_pipeline(
    pipeline=logistic_regression_trained_pipeline,
    X_train=X_train,
    X_test=X_test
)

In [ ]:
metrics = print_test_metrics(
    y_test=y_test,
    test_predictions=test_predictions,
    test_probabilities=test_probabilities
)


In [ ]:
print_confusion_matrix(
    y_test=y_test,
    test_predictions=test_predictions,
    model_name="Logistic Regression"
)

In [ ]:
plot_roc_curve(
    y_test=y_test,
    test_probabilities=test_probabilities,
    model_name="Logistic Regression"
)

In [ ]:
compare_train_vs_test(
    y_train=y_train,
    train_predictions=train_predictions,
    y_test=y_test,
    test_predictions=test_predictions
)

### 3. Decision Tree:

In [ ]:
decision_tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", DecisionTreeClassifier(
        random_state=42
    ))
])

In [ ]:
cv_results = perform_cross_validation(
    pipeline=decision_tree_pipeline,
    X_train=X_train,
    y_train=y_train
)

In [ ]:
print_cv_fold_results(cv_results)

In [ ]:
print_cv_average_results(cv_results)

In [ ]:
print_cv_standard_deviation(cv_results)

In [ ]:
decision_tree_trained_pipeline, training_time = fit_pipeline(
    pipeline=decision_tree_pipeline,
    X_train=X_train,
    y_train=y_train
)

print(f"Training Time: {training_time:.4f} seconds")

In [ ]:
joblib.dump(decision_tree_trained_pipeline, "../outputs/models/decision_tree.pkl")

In [ ]:
plot_feature_importance(
    pipeline=decision_tree_trained_pipeline,
    feature_names=X_train.columns,
    model_name="Decision Tree"
)

In [ ]:
(
    train_predictions,
    test_predictions,
    train_probabilities,
    test_probabilities
) = predict_pipeline(
    pipeline=decision_tree_trained_pipeline,
    X_train=X_train,
    X_test=X_test
)

In [ ]:
_ = print_test_metrics(
    y_test=y_test,
    test_predictions=test_predictions,
    test_probabilities=test_probabilities
)

In [ ]:
print_confusion_matrix(
    y_test=y_test,
    test_predictions=test_predictions,
    model_name="Decision Tree"
)

In [ ]:
plot_roc_curve(
    y_test=y_test,
    test_probabilities=test_probabilities,
    model_name="Decision Tree"
)

In [ ]:
comparison_df = compare_train_vs_test(
    y_train=y_train,
    train_predictions=train_predictions,
    y_test=y_test,
    test_predictions=test_predictions
)

comparison_df

### 4. XgBoost:

In [ ]:
xgboost_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

In [ ]:
cv_results = perform_cross_validation(
    pipeline=xgboost_pipeline,
    X_train=X_train,
    y_train=y_train
)

In [ ]:
print_cv_fold_results(cv_results)

In [ ]:
print_cv_average_results(cv_results)

In [ ]:
print_cv_standard_deviation(cv_results)

In [ ]:
xgboost_trained_pipeline, training_time = fit_pipeline(
    pipeline=xgboost_pipeline,
    X_train=X_train,
    y_train=y_train
)

print(f"Training Time: {training_time:.4f} seconds")

In [ ]:
joblib.dump(xgboost_trained_pipeline, "../outputs/models/xgboost.pkl")

In [ ]:
plot_feature_importance(
    pipeline=xgboost_trained_pipeline,
    feature_names=X_train.columns,
    model_name="XGBoost"
)

In [ ]:
(
    train_predictions,
    test_predictions,
    train_probabilities,
    test_probabilities
) = predict_pipeline(
    pipeline=xgboost_trained_pipeline,
    X_train=X_train,
    X_test=X_test
)

In [ ]:
_ = print_test_metrics(
    y_test=y_test,
    test_predictions=test_predictions,
    test_probabilities=test_probabilities
)

In [ ]:
print_confusion_matrix(
    y_test=y_test,
    test_predictions=test_predictions,
    model_name="XGBoost"
)

In [ ]:
plot_roc_curve(
    y_test=y_test,
    test_probabilities=test_probabilities,
    model_name="XGBoost"
)

In [ ]:
comparison_df = compare_train_vs_test(
    y_train=y_train,
    train_predictions=train_predictions,
    y_test=y_test,
    test_predictions=test_predictions
)

comparison_df

### 5. Gradient Boosting:

In [ ]:
gradient_boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", GradientBoostingClassifier(
        random_state=42
    ))
])

In [ ]:
cv_results = perform_cross_validation(
    pipeline=gradient_boosting_pipeline,
    X_train=X_train,
    y_train=y_train
)

In [ ]:
print_cv_fold_results(cv_results)

In [ ]:
print_cv_average_results(cv_results)

In [ ]:
gradient_boosting_trained_pipeline, training_time = fit_pipeline(
    pipeline=gradient_boosting_pipeline,
    X_train=X_train,
    y_train=y_train
)

print(f"Training Time: {training_time:.4f} seconds")

In [ ]:
joblib.dump(gradient_boosting_trained_pipeline, "../outputs/models/gradient_boosting.pkl")

In [ ]:
plot_feature_importance(
    pipeline=gradient_boosting_trained_pipeline,
    feature_names=X_train.columns,
    model_name="Gradient Boosting"
)

In [ ]:
(
    train_predictions,
    test_predictions,
    train_probabilities,
    test_probabilities
) = predict_pipeline(
    pipeline=gradient_boosting_trained_pipeline,
    X_train=X_train,
    X_test=X_test
)

In [ ]:
_ = print_test_metrics(
    y_test=y_test,
    test_predictions=test_predictions,
    test_probabilities=test_probabilities
)

In [ ]:
print_confusion_matrix(
    y_test=y_test,
    test_predictions=test_predictions,
    model_name="Gradient Boosting"
)

In [ ]:
plot_roc_curve(
    y_test=y_test,
    test_probabilities=test_probabilities,
    model_name="Gradient Boosting"
)

In [ ]:
comparison_df = compare_train_vs_test(
    y_train=y_train,
    train_predictions=train_predictions,
    y_test=y_test,
    test_predictions=test_predictions
)

comparison_df

### 6. Support Vector Machine(SVM):

In [ ]:
svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", SVC(
        probability=True,
        random_state=42
    ))
])

In [ ]:
cv_results = perform_cross_validation(
    pipeline=svm_pipeline,
    X_train=X_train,
    y_train=y_train
)

In [ ]:
print_cv_fold_results(cv_results)

In [ ]:
print_cv_average_results(cv_results)

In [ ]:
print_cv_standard_deviation(cv_results)

In [ ]:
svm_trained_pipeline, training_time = fit_pipeline(
    pipeline=svm_pipeline,
    X_train=X_train,
    y_train=y_train
)

print(f"Training Time: {training_time:.4f} seconds")

In [ ]:
joblib.dump(svm_trained_pipeline, "../outputs/models/svm.pkl")

In [ ]:
(
    train_predictions,
    test_predictions,
    train_probabilities,
    test_probabilities
) = predict_pipeline(
    pipeline=svm_trained_pipeline,
    X_train=X_train,
    X_test=X_test
)

In [ ]:
_= print_test_metrics(
    y_test=y_test,
    test_predictions=test_predictions,
    test_probabilities=test_probabilities
)

In [ ]:
print_confusion_matrix(
    y_test=y_test,
    test_predictions=test_predictions,
    model_name="Support Vector Machine"
)

In [ ]:
plot_roc_curve(
    y_test=y_test,
    test_probabilities=test_probabilities,
    model_name="Support Vector Machine"
)

In [ ]:
comparison_df = compare_train_vs_test(
    y_train=y_train,
    train_predictions=train_predictions,
    y_test=y_test,
    test_predictions=test_predictions
)

comparison_df

### 7. KNN 

In [ ]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", KNeighborsClassifier(
        n_neighbors=5
    ))
])

In [ ]:
cv_results = perform_cross_validation(
    pipeline=knn_pipeline,
    X_train=X_train,
    y_train=y_train
)

In [ ]:
print_cv_fold_results(cv_results)

In [ ]:
print_cv_average_results(cv_results)

In [ ]:
print_cv_standard_deviation(cv_results)

In [ ]:
print_cv_standard_deviation(cv_results)

In [ ]:
knn_trained_pipeline, training_time = fit_pipeline(
    pipeline=knn_pipeline,
    X_train=X_train,
    y_train=y_train
)

print(f"Training Time: {training_time:.4f} seconds")

In [ ]:
joblib.dump(knn_trained_pipeline, "../outputs/models/knn.pkl")

In [ ]:
(
    train_predictions,
    test_predictions,
    train_probabilities,
    test_probabilities
) = predict_pipeline(
    pipeline=knn_trained_pipeline,
    X_train=X_train,
    X_test=X_test
)

In [ ]:
_ = print_test_metrics(
    y_test=y_test,
    test_predictions=test_predictions,
    test_probabilities=test_probabilities
)

In [ ]:
print_confusion_matrix(
    y_test=y_test,
    test_predictions=test_predictions,
    model_name="K-Nearest Neighbors"
)

In [ ]:
plot_roc_curve(
    y_test=y_test,
    test_probabilities=test_probabilities,
    model_name="K-Nearest Neighbors"
)

In [ ]:
comparison_df = compare_train_vs_test(
    y_train=y_train,
    train_predictions=train_predictions,
    y_test=y_test,
    test_predictions=test_predictions
)

comparison_df

## Comparison of all the models:

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Dictionary containing all trained models
models = {
    "Logistic Regression": logistic_regression_trained_pipeline,
    "Decision Tree": decision_tree_trained_pipeline,
    "Random Forest": random_forest_trained_pipeline,
    "Gradient Boosting": gradient_boosting_trained_pipeline,
    "XGBoost": xgboost_trained_pipeline,
    "SVM": svm_trained_pipeline,
    "KNN": knn_trained_pipeline
}

# Store results
results = []

for model_name, model in models.items():

    # Predictions
    y_pred = model.predict(X_test)

    # Probabilities for ROC-AUC
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

# Convert to DataFrame
comparison_df = pd.DataFrame(results)

# Round values to 4 decimal places
comparison_df = comparison_df.round(4)

# Display
comparison_df

## Training and Testing Results comparison:

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

models = {
    "Logistic Regression": logistic_regression_trained_pipeline,
    "Decision Tree": decision_tree_trained_pipeline,
    "Random Forest": random_forest_trained_pipeline,
    "Gradient Boosting": gradient_boosting_trained_pipeline,
    "XGBoost": xgboost_trained_pipeline,
    "SVM": svm_trained_pipeline,
    "KNN": knn_trained_pipeline
}

results = []

for model_name, model in models.items():

    # =========================
    # Training Predictions
    # =========================
    train_pred = model.predict(X_train)
    train_prob = model.predict_proba(X_train)[:, 1]

    # =========================
    # Test Predictions
    # =========================
    test_pred = model.predict(X_test)
    test_prob = model.predict_proba(X_test)[:, 1]

    # =========================
    # Calculate Metrics
    # =========================
    train_accuracy = accuracy_score(y_train, train_pred)
    test_accuracy = accuracy_score(y_test, test_pred)

    train_precision = precision_score(y_train, train_pred)
    test_precision = precision_score(y_test, test_pred)

    train_recall = recall_score(y_train, train_pred)
    test_recall = recall_score(y_test, test_pred)

    train_f1 = f1_score(y_train, train_pred)
    test_f1 = f1_score(y_test, test_pred)

    train_auc = roc_auc_score(y_train, train_prob)
    test_auc = roc_auc_score(y_test, test_prob)

    results.append({
        "Model": model_name,

        "Train Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Accuracy Gap": abs(train_accuracy - test_accuracy),

        "Train Precision": train_precision,
        "Test Precision": test_precision,
        "Precision Gap": abs(train_precision - test_precision),

        "Train Recall": train_recall,
        "Test Recall": test_recall,
        "Recall Gap": abs(train_recall - test_recall),

        "Train F1": train_f1,
        "Test F1": test_f1,
        "F1 Gap": abs(train_f1 - test_f1),

        "Train ROC-AUC": train_auc,
        "Test ROC-AUC": test_auc,
        "ROC-AUC Gap": abs(train_auc - test_auc)
    })

comparison_df = pd.DataFrame(results)

# Round all numeric columns
comparison_df = comparison_df.round(4)

comparison_df

### Selection of best model:

I am considering <b>mainly</b> these metrics for the model selection:

1. <b>Recall:</b> I have learnt that in medical field, it is dangerous that if someone has a disease and model predicts that the patient has no disease. So, the rate of false negatives(recall) should be less.
2. <b>Overfitting:</b> If a model overfits, then it will perform poorly on entirely new data.
3. <b>Accuracy:</b> It shows overall how the model is performing.

Based on these matrices, I have concluded that <b>Logistic Regression</b> is the best model.

#### Reasons:

<b>Recall</b>: Logistic Regression has highest recall(98.59%).

<b>Overfitting:</b> It has the lowest train-test gap. Mainly in the case of Recall, the gap between recall during training is 0.0071 and in the case of Accuracy, its 0.0066. Same goes with precision and f1-score, the training and testing gap is lowest.

### Hyperparameter Tuning using GridSearch CV:

##### Initialization of parameter grid:

In [ ]:
param_grid = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "classifier__penalty": ["l1", "l2"],
    "classifier__solver": ["liblinear"],
    "classifier__class_weight": [None, "balanced"],
    "classifier__max_iter": [100, 300, 500]
}

##### Grid Search:

In [ ]:
grid_search = GridSearchCV(
    estimator=logistic_regression_trained_pipeline,          # Logistic Regression pipeline
    param_grid=param_grid,
    scoring="recall",            # Prioritize recall for breast cancer detection
    cv=5,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search.fit(X_train, y_train)

##### Best Parameters:

In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

##### Best pipeline:

In [ ]:
best_logistic_regression_pipeline = grid_search.best_estimator_
best_logistic_regression_pipeline

##### Evaluating hypermater tuned pipeline:

In [ ]:
# Make predictions
y_pred = best_pipeline.predict(X_test)

# Prediction probabilities
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

# Print metrics
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

##### Classification Report:

In [ ]:
print(classification_report(y_test, y_pred))

##### Confusion Matrix:

In [ ]:
print(confusion_matrix(y_test, y_pred))

### Final Decision after Hyperparameter tuning:

After performing hyperparameter tuning on the best performing model, I analyzed the results, I found out that after hyperparameter tuning, the performance of the model in every metric decrased, so I am deciding to go with the default logistic regression pipeline.